ChatGPT advised that the primary prediction unit is O*NET-SOC Code. I will look for CSVs containing those first.

In [139]:
import pandas as pd

In [140]:
df_occupation_data = pd.read_csv('ONET data/Occupation Data.csv')
df_task_statements = pd.read_csv('ONET data/Task Statements.csv')
df_skills = pd.read_csv('ONET data/Skills.csv')
df_abilities = pd.read_csv('ONET data/Abilities.csv')
df_work_activities = pd.read_csv('ONET data/Work Activities.csv')
df_knowledge = pd.read_csv('ONET data/Knowledge.csv')
df_work_context = pd.read_csv('ONET data/Work Context.csv')
df_tools_used = pd.read_csv('ONET data/Tools Used.csv')
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')

In [141]:
dfs = {
    "occupation_data": df_occupation_data,
    "task_statements": df_task_statements,
    "skills": df_skills,
    "abilities": df_abilities,
    "work_activities": df_work_activities,
    "knowledge": df_knowledge,
    "work_context": df_work_context,
    "tools_used": df_tools_used,
    "technology_skills": df_technology_skills
}

In [142]:
df_task_statements

,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,94.0,08/2023,Incumbent
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,Core,95.0,08/2023,Incumbent
...,...,...,...,...,...,...,...,...
18791,53-7121.00,"Tank Car, Truck, and Ship Loaders",12807,Unload cars containing liquids by connecting h...,Supplemental,85.0,08/2019,Incumbent
18792,53-7121.00,"Tank Car, Truck, and Ship Loaders",12804,"Clean interiors of tank cars or tank trucks, u...",Supplemental,85.0,08/2019,Incumbent
18793,53-7121.00,"Tank Car, Truck, and Ship Loaders",12803,Lower gauge rods into tanks or read meters to ...,Supplemental,85.0,08/2019,Incumbent
18794,53-7121.00,"Tank Car, Truck, and Ship Loaders",12805,Operate conveyors and equipment to transfer gr...,Supplemental,85.0,08/2019,Incumbent


In [143]:
df_occupation_data['O*NET-SOC Code'].nunique()

1016

In [144]:
df_task_statements['O*NET-SOC Code'].nunique()

923

In [145]:
df_skills['O*NET-SOC Code'].nunique()

894

In [146]:
df_abilities['O*NET-SOC Code'].nunique()

894

In [147]:
df_work_activities['O*NET-SOC Code'].nunique()

894

In [148]:
df_knowledge['O*NET-SOC Code'].nunique()

894

In [149]:
df_work_context['O*NET-SOC Code'].nunique()

894

In [150]:
df_tools_used['O*NET-SOC Code'].nunique()

902

In [151]:
df_technology_skills['O*NET-SOC Code'].nunique()

923

# Find Missing

In [152]:
soc_sets = {
    name: set(df['O*NET-SOC Code'].unique())
    for name, df in dfs.items()
}


In [153]:
all_socs = sorted(set.union(*soc_sets.values()))


In [154]:
import pandas as pd

coverage_df = pd.DataFrame({
    name: [soc in soc_sets[name] for soc in all_socs]
    for name in soc_sets
}, index=all_socs)

coverage_df.index.name = 'O*NET-SOC Code'
coverage_df = coverage_df.astype(int)

coverage_df['MISSING_COUNT'] = (
    len(coverage_df.columns) - coverage_df.sum(axis=1)
)

coverage_df

,occupation_data,task_statements,skills,abilities,work_activities,knowledge,work_context,tools_used,technology_skills,MISSING_COUNT
O*NET-SOC Code,,,,,,,,,,
11-1011.00,1,1,1,1,1,1,1,1,1,0
11-1011.03,1,1,1,1,1,1,1,1,1,0
11-1021.00,1,1,1,1,1,1,1,1,1,0
11-1031.00,1,1,0,0,0,0,0,1,1,5
11-2011.00,1,1,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...
55-3014.00,1,0,0,0,0,0,0,0,0,8
55-3015.00,1,0,0,0,0,0,0,0,0,8
55-3016.00,1,0,0,0,0,0,0,0,0,8


In [155]:
coverage_df['MISSING_COUNT'].value_counts()

MISSING_COUNT
0    887
8     93
5     15
6     14
1      7
Name: count, dtype: int64

In [156]:
data_columns = coverage_df.columns.drop('MISSING_COUNT')

total_socs = len(coverage_df)

for col in data_columns:
    missing = (coverage_df[col] == 0).sum()
    print(f"{col}: {missing} jobs missing out of {total_socs}")

occupation_data: 0 jobs missing out of 1016
task_statements: 93 jobs missing out of 1016
skills: 122 jobs missing out of 1016
abilities: 122 jobs missing out of 1016
work_activities: 122 jobs missing out of 1016
knowledge: 122 jobs missing out of 1016
work_context: 122 jobs missing out of 1016
tools_used: 114 jobs missing out of 1016
technology_skills: 93 jobs missing out of 1016


# Narrowing Focus - Technology.

In [157]:
df_technology_skills.columns

Index(['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code',
       'Commodity Title', 'Hot Technology', 'In Demand'],
      dtype='object')

In [158]:
df_technology_skills

,O*NET-SOC Code,Title,Example,Commodity Code,Commodity Title,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,43232202,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,43232306,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,43232201,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,43232303,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,43231601,Accounting software,N,N
...,...,...,...,...,...,...,...
32768,53-7121.00,"Tank Car, Truck, and Ship Loaders",Linux,43233004,Operating system software,Y,N
32769,53-7121.00,"Tank Car, Truck, and Ship Loaders",Microsoft Excel,43232110,Spreadsheet software,Y,N
32770,53-7121.00,"Tank Car, Truck, and Ship Loaders",Microsoft Office software,43231513,Office suite software,Y,N
32771,53-7121.00,"Tank Car, Truck, and Ship Loaders",SAP software,43231602,Enterprise resource planning ERP software,Y,N


In [159]:
df_technology_skills.columns

Index(['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code',
       'Commodity Title', 'Hot Technology', 'In Demand'],
      dtype='object')

In [160]:
df_technology_skills['O*NET-SOC Code'].nunique()

923

In [161]:
df_technology_skills['Title'].nunique()

923

In [162]:
df_technology_skills['Example'].nunique()

8785

In [163]:
df_technology_skills['Commodity Code'].nunique()

137

In [164]:
df_technology_skills['Commodity Title'].nunique()

137

# Tech Skills - Software skills aggregated by job

In [216]:
# Create dataframe titles_skills_and_commods, with aggregates about software for each Title

# Reset DF definition
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')

# Reformat Y/N columns to 1/0, change to binary
df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].replace('Y', 1)
df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].replace('N', 0)
df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].astype(int)

df_technology_skills['In Demand'] = df_technology_skills['In Demand'].replace('Y', 1)
df_technology_skills['In Demand'] = df_technology_skills['In Demand'].replace('N', 0)
df_technology_skills['In Demand'] = df_technology_skills['In Demand'].astype(int)


# Find count of unique skills by title
total_skills_by_title = (
    df_technology_skills
    .groupby(['O*NET-SOC Code', 'Title'])['Example']
    .nunique()
    .reset_index(name='TOTAL_EXAMPLES')
    .sort_values('TOTAL_EXAMPLES', ascending=False)
)

# Find count of unique commodities by title
total_commods_by_title = (
    df_technology_skills
    .groupby(['O*NET-SOC Code', 'Title'])['Commodity Code']
    .nunique()
    .reset_index(name='TOTAL_COMMODITIES')
    .sort_values('TOTAL_COMMODITIES', ascending=False)
)

# Merge
titles_skills_and_commods = pd.merge(total_skills_by_title, total_commods_by_title, on=['O*NET-SOC Code', 'Title'], how='inner')

# Create column list of all unique examples by title
examples_by_title = (
    df_technology_skills
    .groupby('Title')['Example']
    .apply(lambda x: '; '.join(
        f'"{e}"' for e in sorted(x.unique())
    ))
    .reset_index(name='EXAMPLE_LIST')
)

# Merge
titles_skills_and_commods = titles_skills_and_commods.merge(
    examples_by_title,
    on='Title',
    how='left'
)

# Create column list of all unique commodities by title
commods_by_title = (
    df_technology_skills
    .groupby('Title')['Commodity Title']
    .apply(lambda x: '; '.join(
        f'"{e}"' for e in sorted(x.unique())
    ))
    .reset_index(name='COMMODITY_LIST')
)

# Merge
titles_skills_and_commods = titles_skills_and_commods.merge(
    commods_by_title,
    on='Title',
    how='left'
)


# For each job, find the percentage of software used that is considered Hot Technology
title_example_hot = (
    df_technology_skills
    .groupby(['Title', 'Example'])['Hot Technology']
    .max()   # if any row is 1 → result is 1
    .reset_index()
)

hot_percentage = (
    title_example_hot
    .groupby('Title')['Hot Technology']
    .agg(['mean', 'sum', 'count'])
    .reset_index()
)

hot_percentage = hot_percentage.rename(columns={
    'mean': 'PCT_HOT_TECHNOLOGY',
    'sum': 'NUM_HOT_EXAMPLES',
    'count': 'TOTAL_EXAMPLES'
})

hot_percentage['PCT_HOT_TECHNOLOGY'] *= 100

titles_skills_and_commods = titles_skills_and_commods.merge(
    hot_percentage,
    on='Title',
    how='left'
)



# For each job, find the percentage of software used that is considered In Demand
title_example_demand = (
    df_technology_skills
    .groupby(['Title', 'Example'])['In Demand']
    .max()   # if any row is 1 → result is 1
    .reset_index()
)

demand_percentage = (
    title_example_demand
    .groupby('Title')['In Demand']
    .agg(['mean', 'sum', 'count'])
    .reset_index()
)

demand_percentage = demand_percentage.rename(columns={
    'mean': 'PCT_IN_DEMAND',
    'sum': 'NUM_IN_DEMAND_EXAMPLES',
    'count': 'TOTAL_EXAMPLES'
})

demand_percentage['PCT_IN_DEMAND'] *= 100

titles_skills_and_commods = titles_skills_and_commods.merge(
    demand_percentage,
    on='Title',
    how='left'
)


# Merging cause column name wonkiness - drop redundant columns, rename original
titles_skills_and_commods = titles_skills_and_commods.drop(
    columns=['TOTAL_EXAMPLES_y', 'TOTAL_EXAMPLES']
)

titles_skills_and_commods = titles_skills_and_commods.rename(
    columns={'TOTAL_EXAMPLES_x': 'TOTAL_EXAMPLES'}
)


titles_skills_and_commods

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_13361/2226831198.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].replace('N', 0)
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_13361/2226831198.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_technology_skills['In Demand'] = df_technology_skills['In Demand'].replace('N', 0)


,O*NET-SOC Code,Title,TOTAL_EXAMPLES,TOTAL_COMMODITIES,EXAMPLE_LIST,COMMODITY_LIST,PCT_HOT_TECHNOLOGY,NUM_HOT_EXAMPLES,PCT_IN_DEMAND,NUM_IN_DEMAND_EXAMPLES
0,15-1252.00,Software Developers,429,67,"""3M Post-it App""; ""A programming language APL""...","""Access software""; ""Administration software""; ...",34.498834,148,7.692308,33
1,15-1253.00,Software Quality Assurance Analysts and Testers,428,68,"""3M Post-it App""; ""A programming language APL""...","""Access software""; ""Accounting software""; ""Adm...",28.971963,124,5.140187,22
2,15-1299.09,Information Technology Project Managers,334,62,"""24SevenOffice Project""; ""3M Post-it App""; ""AE...","""Access software""; ""Accounting software""; ""Ana...",36.227545,121,2.694611,9
3,15-1243.00,Database Architects,324,59,"""3M Post-it App""; ""ADO.NET""; ""AJAX""; ""ASG Tech...","""Access software""; ""Administration software""; ...",34.567901,112,4.012346,13
4,15-1211.00,Computer Systems Analysts,313,68,"""3M Post-it App""; ""ADP Workforce Now""; ""AJAX"";...","""Access software""; ""Accounting software""; ""Adm...",38.019169,119,3.514377,11
...,...,...,...,...,...,...,...,...,...,...
918,35-9021.00,Dishwashers,2,2,"""Facebook""; ""Microsoft Windows""","""Operating system software""; ""Web page creatio...",100.000000,2,0.000000,0
919,53-7041.00,Hoist and Winch Operators,2,2,"""Microsoft Excel""; ""Microsoft Word""","""Spreadsheet software""; ""Word processing softw...",100.000000,2,0.000000,0
920,53-4041.00,Subway and Streetcar Operators,2,2,"""Microsoft Office software""; ""Word processing ...","""Office suite software""; ""Word processing soft...",50.000000,1,0.000000,0
921,51-4023.00,"Rolling Machine Setters, Operators, and Tender...",2,2,"""Email software""; ""Web browser software""","""Electronic mail software""; ""Internet browser ...",0.000000,0,0.000000,0


In [219]:
titles_skills_and_commods.to_csv('transformed data/Titles software and types.csv', index=False)

# Aggregates by Software.
Confirmed that each 'EXAMPLE' (software name) is only associated with one 'COMMODITY TITLE' (software type).

In [226]:
# Reset DF definition
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')

# Reformat Y/N columns to 1/0, change to binary
df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].replace('Y', 1)
df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].replace('N', 0)
df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].astype(int)

df_technology_skills['In Demand'] = df_technology_skills['In Demand'].replace('Y', 1)
df_technology_skills['In Demand'] = df_technology_skills['In Demand'].replace('N', 0)
df_technology_skills['In Demand'] = df_technology_skills['In Demand'].astype(int)



/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_13361/3719657155.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_technology_skills['Hot Technology'] = df_technology_skills['Hot Technology'].replace('N', 0)
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_13361/3719657155.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_technology_skills['In Demand'] = df_technology_skills['In Demand'].replace('N', 0)


In [227]:
jobs_per_software = (
    df_technology_skills
    .groupby(['Example'])['Title']
    .nunique()
    .reset_index(name='TOTAL_JOBS_PER_SOFTWARE')
    .sort_values('TOTAL_JOBS_PER_SOFTWARE', ascending=False)
)

jobs_per_soft_type = (
    df_technology_skills
    .groupby(['Commodity Title'])['Title']
    .nunique()
    .reset_index(name='TOTAL_JOBS_PER_SOFTWARE_TYPE')
    .sort_values('TOTAL_JOBS_PER_SOFTWARE_TYPE', ascending=False)
)

murjd_df = df_technology_skills.merge(
    jobs_per_software,
    on='Example',
    how='left'
)

murjd_df = murjd_df.merge(
    jobs_per_soft_type,
    on='Commodity Title',
    how='left'
)

software_examples_df = murjd_df[['Example', 'Commodity Title', 'Commodity Code', 'Hot Technology', 'In Demand', 'TOTAL_JOBS_PER_SOFTWARE', 'TOTAL_JOBS_PER_SOFTWARE_TYPE']]
software_examples_df = software_examples_df.drop_duplicates(subset=['Example'], keep='first')

software_examples_df

,Example,Commodity Title,Commodity Code,Hot Technology,In Demand,TOTAL_JOBS_PER_SOFTWARE,TOTAL_JOBS_PER_SOFTWARE_TYPE
0,Adobe Acrobat,Document management software,43232202,1,0,165,293
1,AdSense Tracker,Data base user interface and query software,43232306,0,0,2,632
2,Atlassian JIRA,Content workflow software,43232201,1,0,53,46
3,Blackbaud The Raiser's Edge,Customer relationship management CRM software,43232303,0,0,44,155
4,ComputerEase construction accounting software,Accounting software,43231601,0,0,3,178
...,...,...,...,...,...,...,...
32756,AMCS Platform,Analytical or scientific software,43232605,0,0,1,372
32758,Dossier software,Data base user interface and query software,43232306,0,0,1,632
32761,Mileage logging software,Data base user interface and query software,43232306,0,0,1,632
32763,Routeware software,Map creation software,43233506,0,0,1,87


In [228]:
# Create column list of all unique commodities by (Job)title
jobs_by_software = (
    df_technology_skills
    .groupby('Example')['Title']
    .apply(lambda x: '; '.join(
        f'"{e}"' for e in sorted(x.unique())
    ))
    .reset_index(name='JOB_TITLE_LIST')
)

jobs_by_software

,Example,JOB_TITLE_LIST
0,!Trak-it Solutions !Trak-it HR,"""Compensation and Benefits Managers"""
1,100 Plus Hatch Pattern Library,"""Architectural and Civil Drafters"""
2,1003 Uniform Residential Loan Application,"""Loan Officers"""
3,1099 ProsSoftware,"""Accountants and Auditors"""
4,1CadCam Unigraphics,"""Aerospace Engineers""; ""Automotive Engineers"";..."
...,...,...
8780,xQuery,"""Computer Systems Engineers/Architects"""
8781,xv,"""Physicists"""
8782,yieldWerx,"""Semiconductor Processing Technicians"""
8783,z-Tree,"""Economics Teachers, Postsecondary""; ""Politica..."


In [229]:
# Merge
software_examples_df = software_examples_df.merge(
    jobs_by_software,
    on='Example',
    how='left'
)

software_examples_df

,Example,Commodity Title,Commodity Code,Hot Technology,In Demand,TOTAL_JOBS_PER_SOFTWARE,TOTAL_JOBS_PER_SOFTWARE_TYPE,JOB_TITLE_LIST
0,Adobe Acrobat,Document management software,43232202,1,0,165,293,"""Accountants and Auditors""; ""Administrative La..."
1,AdSense Tracker,Data base user interface and query software,43232306,0,0,2,632,"""Chief Executives""; ""Marketing Managers"""
2,Atlassian JIRA,Content workflow software,43232201,1,0,53,46,"""Administrative Services Managers""; ""Aerospace..."
3,Blackbaud The Raiser's Edge,Customer relationship management CRM software,43232303,0,0,44,155,"""Accountants and Auditors""; ""Administrative Se..."
4,ComputerEase construction accounting software,Accounting software,43231601,0,0,3,178,"""Bookkeeping, Accounting, and Auditing Clerks""..."
...,...,...,...,...,...,...,...,...
8780,AMCS Platform,Analytical or scientific software,43232605,0,0,1,372,"""Refuse and Recyclable Material Collectors"""
8781,Dossier software,Data base user interface and query software,43232306,0,0,1,632,"""Refuse and Recyclable Material Collectors"""
8782,Mileage logging software,Data base user interface and query software,43232306,0,0,1,632,"""Refuse and Recyclable Material Collectors"""
8783,Routeware software,Map creation software,43233506,0,0,1,87,"""Refuse and Recyclable Material Collectors"""


In [230]:
software_examples_df.to_csv('transformed data/Software with jobs.csv', index=False)